# Potential Order via Input Scaling — $V_\theta(a\,u_0)$ vs $a$

For a homogeneous potential $V_\theta(u) \sim C\,\|u\|^p$, one has
$$
V_\theta(a\,u_0) = a^{p}\,V_\theta(u_0)\qquad\text{hence}\qquad
\log\bigl|V_\theta(a\,u_0) - V_\theta(0)\bigr| = p\,\log a + \text{const}.
$$
This notebook probes the effective order $p$ of the learned potential by
picking several test initial conditions $u_0$, scaling each by a scalar
$a\in[a_\text{min},a_\text{max}]$, evaluating $V_\theta(a\,u_0)$, and reading off the
log–log slope. We also split the potential into its three learned components
$V_0, V_1, V_2$ so individual terms can be inspected.

Set `DATASET` and `TRAJ_IDXS` below.

In [ ]:
from pathlib import Path

import numpy as np
import rootutils
import torch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from notebooks.component_scaling.helpers import (  # noqa: E402
    DATASET_CONFIG,
    fit_slopes,
    load_model_for_inference,
    load_test_data,
    plot_components_scaling,
    plot_Fmap,
    plot_total_V_scaling,
    plot_V0_scaling,
    run_scaling_sweep,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT: {ROOT}")
print(f"Device: {device}")

In [ ]:
DATASET = "ckdv"
TRAJ_IDXS = [0, 5, 10, 15, 20]

cfg = DATASET_CONFIG[DATASET]
DISPLAY_NAME = cfg["display_name"]
print(f"Dataset: {DATASET}  ({DISPLAY_NAME})  |  trajs={TRAJ_IDXS}")

In [ ]:
run_dir = ROOT / "logs/official/runs" / DATASET / "s_onsagernet"
print(f"Loading s_onsagernet model for '{DATASET}' ...")
model = load_model_for_inference(run_dir, root=ROOT, device=device)
potential = model.dynamics.potential
print(f"Potential type: {type(potential).__name__}")

data_glob = str(ROOT / "data" / cfg["data_subdir"] / "*.hdf5")
test_data, t_coord, x_coord = load_test_data(data_glob)
N_test, T_data, n_vars, Nx = test_data.shape
print(f"Test set: {N_test} trajectories  |  T_data={T_data}, n_vars={n_vars}, Nx={Nx}")

In [ ]:
# ── Scalar sweep a ∈ [A_MIN, A_MAX], log-spaced ──────────────────────────
A_MIN, A_MAX, N_A = 1e-1, 1.2, 50
a_values = np.logspace(np.log10(A_MIN), np.log10(A_MAX), N_A)
print(f"a sweep: {N_A} values in [{A_MIN:.1e}, {A_MAX:.1e}] (log-spaced)")
print(
    f"sample values: {a_values[0]:.2e}, {a_values[N_A // 4]:.2e}, {a_values[N_A // 2]:.2e}, "
    f"{a_values[3 * N_A // 4]:.2e}, {a_values[-1]:.2e}"
)

In [ ]:
u0_batch = test_data[TRAJ_IDXS, 0].to(device)  # (N_traj, n_vars, Nx)
dV = run_scaling_sweep(potential, u0_batch, a_values, n_vars, Nx, device)
print(f"|dV| range: [{np.abs(dV['V']).min():.3e}, {np.abs(dV['V']).max():.3e}]")

In [ ]:
A_FIT_MIN = A_MIN
A_FIT_MAX = A_MAX
fit_mask = (a_values >= A_FIT_MIN) & (a_values <= A_FIT_MAX)
print(f"Slope fit window: a ∈ [{A_FIT_MIN}, {A_FIT_MAX}]  ({fit_mask.sum()} points)")

slopes, intercepts, r2 = {}, {}, {}
for name in ("V", "V0", "V1", "V2"):
    slopes[name], intercepts[name], r2[name] = fit_slopes(a_values, dV[name], fit_mask)

print(f"\n{'#':>4}  {'p(V)':>8}  {'p(V0)':>8}  {'p(V1)':>8}  {'p(V2)':>8}  {'R²(V0)':>8}")
for k in range(len(TRAJ_IDXS)):
    print(
        f"{k + 1:>4d}  {slopes['V'][k]:>8.3f}  {slopes['V0'][k]:>8.3f}  "
        f"{slopes['V1'][k]:>8.3f}  {slopes['V2'][k]:>8.3f}  {r2['V0'][k]:>8.3f}"
    )
print(f"\nMean p(V0) over trajs: {np.nanmean(slopes['V0']):.3f}")

In [ ]:
plot_total_V_scaling(
    a_values,
    dV["V"],
    slopes["V"],
    intercepts["V"],
    traj_idxs=TRAJ_IDXS,
    display_name=DISPLAY_NAME,
    a_min=A_MIN,
    a_max=A_MAX,
    a_fit_max=A_FIT_MAX,
    out_path=ROOT / f"figs/component_scaling/fpu_potential_scaling_order.pdf",
)

In [ ]:
plot_components_scaling(
    a_values,
    dV_by_name={"V_0": dV["V0"], "V_2": dV["V2"]},
    slopes_by_name={"V_0": slopes["V0"], "V_2": slopes["V2"]},
    intercepts_by_name={"V_0": intercepts["V0"], "V_2": intercepts["V2"]},
    r2_by_name={"V_0": r2["V0"], "V_2": r2["V2"]},
    traj_idxs=TRAJ_IDXS,
    display_name=DISPLAY_NAME,
    a_min=A_MIN,
    a_max=A_MAX,
    a_fit_max=A_FIT_MAX,
    out_path=ROOT / f"figs/component_scaling/fpu_potential_scaling_order_components.pdf",
)

In [ ]:
plot_V0_scaling(
    a_values,
    dV0=dV["V0"],
    slopes=slopes["V0"],
    intercepts=intercepts["V0"],
    r2=r2["V0"],
    traj_idxs=TRAJ_IDXS,
    display_name=DISPLAY_NAME,
    a_min=A_MIN,
    a_max=A_MAX,
    a_fit_max=A_FIT_MAX,
    out_path=ROOT / f"figs/component_scaling/fpu_potential_scaling_order_V0.pdf",
)

## Pointwise nonlinear energy density $F(u)$

The learned $V_0$ is

$$V_0(u) = \int \bigl[\tfrac12\,a\,u(x)^2 + F(u(x))\bigr]\,dx,$$

so the pointwise function $F:\mathbb R\to\mathbb R$ captures the nonlinear part
of the potential. We evaluate it on a scalar grid of $u$ values to read off its
shape directly (symmetry, dominant order at large $|u|$, etc.). Works when
`n_vars==1`.

In [ ]:
plot_Fmap(
    potential,
    test_data,
    n_vars=n_vars,
    display_name=DISPLAY_NAME,
    out_path=ROOT / f"figs/component_scaling/fpu_Fmap.pdf",
    ylim=(-16, 16),
    box_ylim=(-5, 5),
    device=device,
)